In [ ]:
import sys
print(sys.executable)

In [ ]:
import requests
import pandas as pd

APP_ID = ""   # 填入你的 Adzuna app_id
APP_KEY = ""  # 填入你的 Adzuna app_key

In [ ]:
url = "https://api.adzuna.com/v1/api/jobs/gb/search/1"
params = {
    "app_id": APP_ID,
    "app_key": APP_KEY,
    "results_per_page": 50,
    "what": "data analyst",
    "where": "london",
}

r = requests.get(url, params=params)
print(r.status_code)

data = r.json()
print(data["count"])

In [ ]:
url = "https://api.adzuna.com/v1/api/jobs/gb/search/2"
params = {
    "app_id": APP_ID,
    "app_key": APP_KEY,
    "results_per_page": 50,
    "what": "data analyst",
    "where": "london",
}

r = requests.get(url, params=params)
print(r.status_code)

data = r.json()
print(data["count"])

In [ ]:
data = r.json()
print(data["results"][0]["title"])
print(data["results"][0]["id"])

In [ ]:
import time

all_jobs = []

for page in range(1, 6):
    url = f"https://api.adzuna.com/v1/api/jobs/gb/search/{page}"
    params = {
        "app_id": APP_ID,
        "app_key": APP_KEY,
        "results_per_page": 50,
        "what": "data analyst",
        "where": "london",
    }
    r = requests.get(url, params=params)
    print(f"page {page}: {r.status_code}")
    
    data = r.json()
    all_jobs.extend(data["results"])
    
    time.sleep(1)

print(f"总共拿到 {len(all_jobs)} 条")

In [ ]:
df = pd.DataFrame(all_jobs)
print(df.shape)
print(df["id"].nunique())

In [ ]:
df = pd.DataFrame(data["results"])
print(df.shape)
df.head()

In [ ]:
print(len(all_jobs))

In [ ]:
df = pd.DataFrame(all_jobs)
print(df.shape)
print(df["id"].nunique())

In [ ]:
def fetch_jobs(keyword, location, max_pages=5):
    """取某个关键词在某地区的岗位，返回列表"""
    jobs = []
    for page in range(1, max_pages + 1):
        url = f"https://api.adzuna.com/v1/api/jobs/gb/search/{page}"
        params = {
            "app_id": APP_ID,
            "app_key": APP_KEY,
            "results_per_page": 50,
            "what": keyword,
            "where": location,
        }
        r = requests.get(url, params=params)
        if r.status_code != 200:
            print(f"  page {page} failed: {r.status_code}")
            break
        
        results = r.json()["results"]
        if len(results) == 0:
            break
        
        for job in results:
            job["search_keyword"] = keyword
        jobs.extend(results)
        
        time.sleep(1)
    return jobs

In [ ]:
test = fetch_jobs("data analyst", "london", max_pages=2)
print(len(test))

In [ ]:
keywords = [
    "data analyst",
    "data scientist",
    "machine learning engineer",
    "insight analyst",
    "graduate analyst",
]

all_jobs = []

for kw in keywords:
    print(f"fetching: {kw}")
    jobs = fetch_jobs(kw, "london", max_pages=5)
    print(f"  got {len(jobs)}")
    all_jobs.extend(jobs)

print(f"\n总计 {len(all_jobs)} 条")

In [ ]:
import os

os.makedirs("../data/raw", exist_ok=True)

df = pd.DataFrame(all_jobs)
df.to_csv("../data/raw/adzuna_london_20260806.csv", index=False)

print(df.shape)
print(df["id"].nunique())

In [ ]:
%pip install pandas